# NB-02: QKV Projections and Multi-Head Layout

## Learning Objectives
- Understand how Q, K, V linear projections transform video tokens -- all three are `dim`-to-`dim` (no expansion), with head splitting done post-projection via einops
- Trace the two multi-head layout conventions: `(B, S, N, D)` for flash_attn_2/3 vs `(B, N, S, D)` for PyTorch SDPA
- Verify lossless round-trip between flat `(B, S, dim)` and multi-head `(B, N, S, D)` layouts

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm -- used for Q/K normalization)
- **Assumed concepts:** Linear layers (`nn.Linear`), tensor reshaping, einops `rearrange`

## Concept Map
- **QKV projections** -> used in SelfAttention (NB-04) and CrossAttention (NB-04)
- **Multi-head layout** -> used in attention computation (NB-04)
- **head_dim = dim // num_heads** -> determines RoPE frequency band sizes (NB-03)

In [1]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import symbols used in this notebook
from diffsynth.models.wan_video_dit import SelfAttention, RMSNorm

import torch
import torch.nn as nn
from einops import rearrange

print("Setup complete.")

Project root: /home/basheer/Signapse/Codes/wan22-architecture-course
Setup complete.


## 1. QKV Projections

In WAN 2.2's `SelfAttention` module, queries (Q), keys (K), and values (V) are each produced by a separate `nn.Linear(dim, dim)` projection -- three independent linear layers with **no dimension expansion**. The output shape is identical to the input shape: `[B, S, dim]`.

This is a deliberate design choice: the head splitting is done **after** the projection, using einops. Contrast this with some implementations that project directly to `[dim // num_heads]` per head -- WAN 2.2 uses full-dim projections and then reshapes.

An important asymmetry: Q and K are additionally normalized by RMSNorm (Q/K norm, lines 142-143) to stabilize attention scores. V is **not** normalized -- its magnitudes carry semantic information that should be preserved.

Source: `diffsynth/models/wan_video_dit.py`, line 125

In [2]:
# Source: diffsynth/models/wan_video_dit.py, line 125
# Reduced config
B, S, dim, num_heads = 1, 20, 384, 4
head_dim = dim // num_heads  # 96

sa = SelfAttention(dim=dim, num_heads=num_heads)
x = torch.randn(B, S, dim)

with torch.no_grad():
    q = sa.norm_q(sa.q(x))  # Project + normalize Q
    k = sa.norm_k(sa.k(x))  # Project + normalize K
    v = sa.v(x)              # Project V (no norm)

assert q.shape == torch.Size([B, S, dim]), f"Q shape: expected {(B, S, dim)}, got {q.shape}"
assert k.shape == torch.Size([B, S, dim]), f"K shape: expected {(B, S, dim)}, got {k.shape}"
assert v.shape == torch.Size([B, S, dim]), f"V shape: expected {(B, S, dim)}, got {v.shape}"
print(f"Q: {q.shape}  (projected + RMSNorm)")
print(f"K: {k.shape}  (projected + RMSNorm)")
print(f"V: {v.shape}  (projected, no norm)")
print(f"head_dim = {dim} // {num_heads} = {head_dim}")

Q: torch.Size([1, 20, 384])  (projected + RMSNorm)
K: torch.Size([1, 20, 384])  (projected + RMSNorm)
V: torch.Size([1, 20, 384])  (projected, no norm)
head_dim = 384 // 4 = 96


## 2. Two Layout Conventions

After the QKV projections, the flat `dim`-sized vectors are split into `num_heads` heads, each of size `head_dim = dim // num_heads`. This is done via a single `einops.rearrange` call -- no data is copied, just the shape metadata changes.

**Two competing conventions exist in the codebase**, driven by different attention backends:

| Convention | Shape | Backend | Axis order |
|------------|-------|---------|------------|
| Sequence-first | `(B, S, N, D)` | flash_attn_2, flash_attn_3 | batch, sequence, heads, head_dim |
| Head-first | `(B, N, S, D)` | PyTorch SDPA, SageAttn | batch, heads, sequence, head_dim |

The `flash_attention` function in WAN 2.2 handles both -- it selects the rearrange pattern based on which backend is available at runtime. On CPU (no flash_attn installed), it falls back to PyTorch SDPA with the `(B, N, S, D)` layout.

The VACE encoder and S2V model both inherit this same dual-backend attention dispatch.

Source: `diffsynth/models/wan_video_dit.py`, line 28

In [3]:
# Source: diffsynth/models/wan_video_dit.py, line 28
# Flash Attention layout: (B, S, N, D) -- sequence-first
q_flash = rearrange(q, "b s (n d) -> b s n d", n=num_heads)
assert q_flash.shape == torch.Size([B, S, num_heads, head_dim])

# SDPA layout: (B, N, S, D) -- head-first
q_sdpa = rearrange(q, "b s (n d) -> b n s d", n=num_heads)
assert q_sdpa.shape == torch.Size([B, num_heads, S, head_dim])

print(f"Flat:    {q.shape}  ->  (B, S, dim)")
print(f"Flash:   {q_flash.shape}  ->  (B, S, N, D)")
print(f"SDPA:    {q_sdpa.shape}  ->  (B, N, S, D)")
print(f"In both cases: N={num_heads} heads, D={head_dim} head_dim")

Flat:    torch.Size([1, 20, 384])  ->  (B, S, dim)
Flash:   torch.Size([1, 20, 4, 96])  ->  (B, S, N, D)
SDPA:    torch.Size([1, 4, 20, 96])  ->  (B, N, S, D)
In both cases: N=4 heads, D=96 head_dim


## 3. Round-Trip Verification

Because head splitting is a pure reshape (no computation, no data movement), merging the heads back must produce the exact original tensor. This is an important property: the attention mechanism operates on the split representation, but the output is merged back before the final output projection `self.o`.

Verifying this round-trip guarantees that einops `rearrange` is being used correctly -- any off-by-one in the head count or dimension ordering would produce a mismatch.

In [4]:
# Round-trip: flat -> multi-head -> flat
q_roundtrip_flash = rearrange(q_flash, "b s n d -> b s (n d)", n=num_heads)
q_roundtrip_sdpa = rearrange(q_sdpa, "b n s d -> b s (n d)", n=num_heads)
assert torch.allclose(q, q_roundtrip_flash), "Flash round-trip should be lossless"
assert torch.allclose(q, q_roundtrip_sdpa), "SDPA round-trip should be lossless"
print("Round-trip verified: both layouts recover original tensor exactly")
print(f"Flash max diff: {(q - q_roundtrip_flash).abs().max().item():.2e}")
print(f"SDPA  max diff: {(q - q_roundtrip_sdpa).abs().max().item():.2e}")

Round-trip verified: both layouts recover original tensor exactly
Flash max diff: 0.00e+00
SDPA  max diff: 0.00e+00


## 4. Q/K Normalization Asymmetry

WAN 2.2 normalizes Q and K with RMSNorm but **not** V. This asymmetry is deliberate:

- **Why normalize Q and K?** Attention scores are computed as dot products between Q and K vectors. If these vectors have unconstrained magnitudes, the dot products can become very large, causing softmax to saturate (all attention on one token). Normalizing Q and K ensures unit-variance inputs, keeping attention scores in a stable range.

- **Why not normalize V?** V carries the actual content that gets weighted and summed. Normalizing it would destroy magnitude information that the model has learned to encode. The output projection `self.o` expects V values at their natural scale.

We can verify this by comparing the standard deviations: Q and K should have std closer to 1.0 (post-normalization), while V's std depends on the random initialization of `self.v`.

In [5]:
# Q and K are normalized, V is not
print(f"Q std (post-norm): {q.std():.4f}")
print(f"K std (post-norm): {k.std():.4f}")
print(f"V std (no norm):   {v.std():.4f}")
print()
# Q and K stds should be closer to 1.0 than V
print("Q and K are normalized by RMSNorm -> std closer to 1.0")
print("V is not normalized -> std depends on projection weights")

Q std (post-norm): 1.0000
K std (post-norm): 1.0000
V std (no norm):   0.5840

Q and K are normalized by RMSNorm -> std closer to 1.0
V is not normalized -> std depends on projection weights


## Summary

### Key Takeaways
- **QKV projections** in WAN 2.2 are dim-to-dim: `nn.Linear(dim, dim)` for Q, K, and V -- no dimension expansion. Head splitting is a post-projection reshape, not a per-head projection.
- **Two head layout conventions** coexist: `(B, S, N, D)` (sequence-first) for flash_attn_2/3 backends and `(B, N, S, D)` (head-first) for PyTorch SDPA and SageAttn. The `flash_attention` function selects between them at runtime.
- **Q/K normalization asymmetry**: Q and K are normalized by RMSNorm to stabilize attention scores. V is left unnormalized to preserve magnitude information for the output.

### Source References
| Symbol | Location |
|--------|----------|
| SelfAttention | `diffsynth/models/wan_video_dit.py`, line 125 |
| flash_attention | `diffsynth/models/wan_video_dit.py`, line 28 |
| RMSNorm (Q/K norm) | `diffsynth/models/wan_video_dit.py`, line 101 |

## Exercises

### Exercise 1 -- Parameter count
Count the total parameters in `SelfAttention`: 4 linear layers (`self.q`, `self.k`, `self.v`, `self.o`, each `dim x dim + dim`) plus 2 RMSNorm layers (`norm_q`, `norm_k`, each `dim` params). Verify your calculation matches `sum(p.numel() for p in sa.parameters())`. With `dim=384`, what is the total?

### Exercise 2 -- Head dimension experiment
Change `num_heads` to 8 (so `head_dim` becomes `384 // 8 = 48`). Verify that all shape assertions still pass with the new head count. Why might more heads with smaller dimensions be useful? (Hint: think about the diversity of attention patterns vs the capacity of each head.)

### Exercise 3 -- Remove Q/K norm
Comment out the `sa.norm_q` and `sa.norm_k` calls in the QKV demo cell, using `q = sa.q(x)` directly. Compare the std of Q and K before and after normalization. What happens to the magnitude of attention scores (Q @ K^T) when Q and K are not normalized?